# Category-Probability and KL-Divergence Filtering

This notebook applies category-based filtering to aspect candidates in two stages:

1. train an SVM category classifier with probability estimates and aggregate category confidence for each candidate aspect;
2. filter candidates using confidence thresholds and KL divergence from a uniform category distribution.

The goal is to remove weak or insufficiently category-specific aspect candidates and evaluate the resulting precision–recall trade-off.


## 1. Setup


In [ ]:
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC


## 2. Category classifier with probability estimates

The source experiment represents each training example as the sentence text concatenated with its annotated/candidate aspect expression and trains a linear SVM with `probability=True`.


In [ ]:
def train_category_probability_classifier(
    training_dataframe,
):
    """Train the CountVectorizer + linear SVM configuration from the source experiment."""
    features = (
        training_dataframe["text"]
        + " "
        + training_dataframe["opinion"]
    )
    labels = training_dataframe["category"]

    (
        X_train,
        X_validation,
        y_train,
        y_validation,
    ) = train_test_split(
        features,
        labels,
        test_size=0.2,
        random_state=42,
    )

    vectorizer = CountVectorizer()

    X_train_vectorized = vectorizer.fit_transform(
        X_train
    )
    X_validation_vectorized = vectorizer.transform(
        X_validation
    )

    classifier = SVC(
        C=1.0,
        kernel="linear",
        probability=True,
        random_state=42,
    )
    classifier.fit(
        X_train_vectorized,
        y_train,
    )

    validation_predictions = classifier.predict(
        X_validation_vectorized
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y_validation,
            validation_predictions,
            average="weighted",
            zero_division=0,
        )
    )

    validation_metrics = {
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

    return (
        vectorizer,
        classifier,
        validation_metrics,
    )


In [ ]:
# `category_training_dataframe` is the sentence + gold-aspect/category
# dataframe constructed from the annotated SemEval training data.
vectorizer, category_classifier, validation_metrics = (
    train_category_probability_classifier(
        category_training_dataframe
    )
)

validation_metrics


## 3. Predict category probabilities for candidate aspects

`candidate_dataframe` contains sentence text and candidate aspect expressions. The trained SVM predicts an aspect category and a probability distribution for each candidate occurrence.


In [ ]:
# `candidate_dataframe` contains sentence text and the unfiltered
# aspect candidates produced by the preceding extraction stage.
candidate_predictions = (
    predict_candidate_category_probabilities(
        candidate_dataframe,
        vectorizer,
        category_classifier,
    )
)

candidate_predictions.head()


## 4. Aggregate probabilities by candidate word and category


In [ ]:
def aggregate_word_category_probabilities(
    candidate_predictions,
):
    """Collect and average predicted confidence by candidate word and category."""
    probability_lists = {}

    for _, row in candidate_predictions.iterrows():
        words = (
            row["opinions"].split(",")
            if "," in row["opinions"]
            else [row["opinions"]]
        )

        category = row["predicted_category"]
        probability = row["best_probability"]

        for word in words:
            word = word.strip()

            probability_lists.setdefault(
                word,
                {}
            ).setdefault(
                category,
                []
            ).append(
                probability
            )

    probability_lists = {
        word: dict(
            sorted(
                category_probabilities.items()
            )
        )
        for word, category_probabilities
        in probability_lists.items()
    }

    mean_probabilities = {
        word: {
            category: np.mean(probabilities)
            for category, probabilities
            in category_probabilities.items()
        }
        for word, category_probabilities
        in probability_lists.items()
    }

    most_probable_categories = {
        word: max(
            category_probabilities,
            key=category_probabilities.get,
        )
        for word, category_probabilities
        in mean_probabilities.items()
    }

    return (
        probability_lists,
        mean_probabilities,
        most_probable_categories,
    )


(
    word_category_probability_lists,
    mean_category_probabilities,
    most_probable_categories,
) = aggregate_word_category_probabilities(
    candidate_predictions
)


In [ ]:
def group_words_by_category(
    most_probable_categories,
    mean_probabilities,
):
    """Group candidate words by their most probable category and sort by confidence."""
    grouped_categories = defaultdict(list)

    for word, category in (
        most_probable_categories.items()
    ):
        grouped_categories[category].append(
            (
                word,
                mean_probabilities[word][category],
            )
        )

    return {
        category: sorted(
            words,
            key=lambda item: item[1],
            reverse=True,
        )
        for category, words
        in grouped_categories.items()
    }


words_by_category = group_words_by_category(
    most_probable_categories,
    mean_category_probabilities,
)

words_by_category


## 5. Confidence-based candidate filtering

Category probabilities are aggregated across repeated occurrences of each candidate. Candidate-removal thresholds from `0.85` to `0.30` are evaluated to measure their effect on precision, recall, and F1.


In [ ]:
def calculate_weighted_candidate_scores(
    word_category_probability_lists,
):
    """Calculate occurrence-weighted category-confidence scores."""
    weighted_means = defaultdict(dict)

    for word, category_probabilities in (
        word_category_probability_lists.items()
    ):
        for category, probabilities in (
            category_probabilities.items()
        ):
            weights = np.arange(
                1,
                len(probabilities) + 1,
            )

            weighted_means[word][category] = (
                np.average(
                    probabilities,
                    weights=weights,
                )
            )

    candidate_scores = {}

    for word, category_scores in (
        weighted_means.items()
    ):
        sorted_categories = sorted(
            category_scores,
            key=category_scores.get,
            reverse=True,
        )

        for category in sorted_categories:
            candidate_scores[word] = (
                category_scores[category]
            )

    return dict(weighted_means), candidate_scores


(
    weighted_category_means,
    candidate_confidence_scores,
) = calculate_weighted_candidate_scores(
    word_category_probability_lists
)


In [ ]:
CONFIDENCE_THRESHOLDS = [
    0.85,
    0.80,
    0.75,
    0.70,
    0.65,
    0.60,
    0.55,
    0.50,
    0.45,
    0.40,
    0.35,
    0.30,
]


def low_confidence_words(
    candidate_scores,
    threshold,
):
    """Return candidates whose score is below the selected confidence threshold."""
    return {
        word: score
        for word, score
        in candidate_scores.items()
        if score < threshold
    }


confidence_filter_sets = {
    threshold: low_confidence_words(
        candidate_confidence_scores,
        threshold,
    )
    for threshold in CONFIDENCE_THRESHOLDS
}


## 6. Evaluate filtering thresholds


In [ ]:
# `candidate_aspects` is the unfiltered candidate list from the
# preceding extraction stage; `gold_aspects` is used only for evaluation.
confidence_filter_results = (
    evaluate_filter_thresholds(
        candidate_aspects,
        gold_aspects,
        confidence_filter_sets,
    )
)

confidence_filter_results


## 7. KL-divergence filtering

For each candidate word, mean category-confidence values are normalized into a probability distribution. KL divergence is calculated against a uniform distribution over the observed categories.

Low KL divergence indicates a less category-specific candidate and can therefore be used as an additional filtering criterion.


In [ ]:
def calculate_kl_divergence_from_uniform(
    mean_probabilities,
):
    """Calculate KL divergence from a uniform category distribution."""
    kl_divergences = {}

    for word, category_probabilities in (
        mean_probabilities.items()
    ):
        probabilities = np.array(
            list(category_probabilities.values()),
            dtype=float,
        )

        normalized_probabilities = (
            probabilities
            / probabilities.sum()
        )

        uniform_distribution = np.ones(
            len(normalized_probabilities)
        ) / len(normalized_probabilities)

        kl_divergences[word] = np.sum(
            normalized_probabilities
            * np.log(
                normalized_probabilities
                / uniform_distribution
            )
        )

    return kl_divergences


candidate_kl_divergences = (
    calculate_kl_divergence_from_uniform(
        mean_category_probabilities
    )
)


In [ ]:
kl_filter_results = evaluate_filter_thresholds(
    candidate_aspects,
    gold_aspects,
    kl_filter_sets,
)

kl_filter_results


## 8. Pipeline role

The confidence and KL experiments above are filtering stages rather than independent aspect extractors. Their inputs come from the preceding propagation-based extraction pipeline.

The thesis reports the category-filtered stage and the subsequent KL-filtered stage as successive precision/recall trade-offs; this notebook keeps the filters separate so their contribution can be inspected directly.
